In [ ]:
# Cell 1: Setup
%pip install ultralytics sahi -q
%pip install huggingface_hub -q

import ultralytics
import torch
import os
import shutil
import yaml
import cv2
import glob
from tqdm.notebook import tqdm

print(f"Setup Complete. YOLO version: {ultralytics.__version__}")

In [ ]:
# Cell 2: SANITIZE & CLONE DATASET
# This fixes "imdecode" errors and "Read-only" errors simultaneously.

# 1. FIND SOURCE DATASET
input_root = '/kaggle/input'
yaml_source = None
for root, dirs, files in os.walk(input_root):
    if 'data.yaml' in files:
        yaml_source = os.path.join(root, 'data.yaml')
        break

if not yaml_source:
    raise FileNotFoundError("Could not find data.yaml in /kaggle/input. Did you add the dataset?")

# 2. DEFINE DESTINATION (Writable Storage)
clean_root = '/kaggle/working/clean_dataset'
if os.path.exists(clean_root):
    shutil.rmtree(clean_root) # Clear previous runs
os.makedirs(clean_root, exist_ok=True)

print(f"🚀 Cloning healthy data from {os.path.dirname(yaml_source)} to {clean_root}...")

# 3. READ CONFIG & PREPARE COPY
with open(yaml_source, 'r') as f:
    config = yaml.safe_load(f)

# Helper function to copy and sanitize
def copy_and_sanitize(src_folder_name, dst_folder_name):
    # Construct absolute source path
    src_path = os.path.join(os.path.dirname(yaml_source), src_folder_name)
    dst_path = os.path.join(clean_root, dst_folder_name)
    
    # Create Dirs
    os.makedirs(dst_path, exist_ok=True) # images
    label_src = src_path.replace('images', 'labels')
    label_dst = dst_path.replace('images', 'labels')
    os.makedirs(label_dst, exist_ok=True) # labels
    
    print(f"   Scanning: {src_path} ...")
    
    valid_count = 0
    bad_count = 0
    
    # Get all images
    files = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp']:
        files.extend(glob.glob(os.path.join(src_path, ext)))
        files.extend(glob.glob(os.path.join(src_path, '**', ext), recursive=True))
        
    for file in tqdm(files, desc=f"Copying {dst_folder_name}"):
        try:
            # CHECK 1: File Size
            if os.path.getsize(file) == 0:
                bad_count += 1
                continue
                
            # CHECK 2: Deep Integrity Check (OpenCV)
            with open(file, "rb") as f:
                bytes_img = f.read()
            import numpy as np
            nparr = np.frombuffer(bytes_img, np.uint8)
            
            if nparr.size == 0: # This catches the "buf is not numpy" error
                bad_count += 1
                continue
                
            img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
            if img is None:
                bad_count += 1
                continue
                
            # IF WE ARE HERE, IMAGE IS GOOD. COPY IT.
            filename = os.path.basename(file)
            shutil.copy2(file, os.path.join(dst_path, filename))
            
            # Copy Label if it exists
            label_name = os.path.splitext(filename)[0] + '.txt'
            label_file = os.path.join(label_src, label_name)
            if os.path.exists(label_file):
                shutil.copy2(label_file, os.path.join(label_dst, label_name))
            
            valid_count += 1
            
        except Exception:
            bad_count += 1
            continue
            
    print(f"   ✅ Copied {valid_count} images. 🗑️ Discarded {bad_count} corrupt files.")
    return dst_path

# 4. EXECUTE COPY
# Note: Adjust these keys if your yaml uses different names
new_train = copy_and_sanitize('train/images', 'train/images')
new_val = copy_and_sanitize('valid/images', 'valid/images')
new_test = copy_and_sanitize('test/images', 'test/images')

# 5. CREATE NEW CONFIG
new_config = config.copy()
new_config['path'] = clean_root
new_config['train'] = 'train/images'
new_config['val'] = 'valid/images'
new_config['test'] = 'test/images'

final_yaml = '/kaggle/working/bigo_clean.yaml'
with open(final_yaml, 'w') as f:
    yaml.dump(new_config, f)

print(f"\n✨ DONE. Clean config saved to: {final_yaml}")

In [ ]:
# Cell 1: RESTORE DATASET & REGENERATE CONFIG
import os
import shutil
import yaml
import cv2
import glob
from tqdm.notebook import tqdm

print("🔄 Restoring and Sanitizing Dataset...")

# 1. FIND SOURCE
input_root = '/kaggle/input'
yaml_source = None
for root, dirs, files in os.walk(input_root):
    if 'data.yaml' in files:
        yaml_source = os.path.join(root, 'data.yaml')
        break

if not yaml_source:
    raise FileNotFoundError("Could not find data.yaml in input. Is the dataset attached?")

# 2. DEFINE DESTINATION
clean_root = '/kaggle/working/clean_dataset'
if os.path.exists(clean_root):
    shutil.rmtree(clean_root) # Clean start
os.makedirs(clean_root, exist_ok=True)

# 3. CLONING FUNCTION
def clone_and_sanitize(src_folder_name, dst_folder_name):
    src_path = os.path.join(os.path.dirname(yaml_source), src_folder_name)
    dst_path = os.path.join(clean_root, dst_folder_name)
    
    # Mirror directory structure
    os.makedirs(dst_path, exist_ok=True) # images
    label_src = src_path.replace('images', 'labels')
    label_dst = dst_path.replace('images', 'labels')
    os.makedirs(label_dst, exist_ok=True) # labels
    
    print(f"   📂 Scanning: {src_path}")
    
    valid = 0
    bad = 0
    
    # Grab all images
    files = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp']:
        files.extend(glob.glob(os.path.join(src_path, ext)))
        
    for file in tqdm(files, desc=f"Cloning {dst_folder_name}"):
        try:
            # INTEGRITY CHECK
            with open(file, "rb") as f:
                bytes_img = f.read()
            
            # 1. Check Zero Byte
            if len(bytes_img) == 0:
                bad += 1
                continue
                
            # 2. Check Numpy Decode (The one that crashed you before)
            import numpy as np
            nparr = np.frombuffer(bytes_img, np.uint8)
            if nparr.size == 0:
                bad += 1
                continue
                
            # 3. Check OpenCV
            img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
            if img is None:
                bad += 1
                continue
                
            # COPY GOOD FILE
            filename = os.path.basename(file)
            shutil.copy2(file, os.path.join(dst_path, filename))
            
            # COPY LABEL
            label_name = os.path.splitext(filename)[0] + '.txt'
            label_file = os.path.join(label_src, label_name)
            if os.path.exists(label_file):
                shutil.copy2(label_file, os.path.join(label_dst, label_name))
                
            valid += 1
            
        except:
            bad += 1
            continue
            
    print(f"   ✅ Processed: {valid} valid, {bad} corrupt removed.")
    return dst_path

# 4. EXECUTE CLONE
# Note: Check if your YAML uses 'train/images' or just 'train'
# We try to guess based on standard Roboflow structure
try:
    with open(yaml_source, 'r') as f:
        raw_config = yaml.safe_load(f)
        
    train_dir = raw_config.get('train', 'train/images')
    if train_dir.startswith('/'): train_dir = train_dir.split('/')[-2] + '/' + train_dir.split('/')[-1]
    
    val_dir = raw_config.get('val', 'valid/images')
    if val_dir.startswith('/'): val_dir = val_dir.split('/')[-2] + '/' + val_dir.split('/')[-1]
    
    test_dir = raw_config.get('test', 'test/images')
    if test_dir.startswith('/'): test_dir = test_dir.split('/')[-2] + '/' + test_dir.split('/')[-1]

    clone_and_sanitize(train_dir, 'train/images')
    clone_and_sanitize(val_dir, 'valid/images')
    clone_and_sanitize(test_dir, 'test/images')

except Exception as e:
    print(f"Path guessing failed ({e}). Defaulting to standard structure...")
    clone_and_sanitize('train/images', 'train/images')
    clone_and_sanitize('valid/images', 'valid/images')
    clone_and_sanitize('test/images', 'test/images')

# 5. CREATE CONFIG
new_config = {
    'path': clean_root,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 4, # Override this with your actual class count if different
    'names': ['Fresh', 'Rotten', 'Sprouted', 'Cuts'] # Override if different
}

final_yaml = '/kaggle/working/bigo_clean_config.yaml'
with open(final_yaml, 'w') as f:
    yaml.dump(new_config, f)

print(f"\n✨ RECOVERY COMPLETE. Config saved to: {final_yaml}")

In [ ]:
# Cell: EMERGENCY MEMORY FLUSH
import torch
import gc

# 1. Delete the model variable if it exists
if 'model' in globals():
    del model

# 2. Force Garbage Collection
gc.collect()

# 3. Clear GPU Cache
torch.cuda.empty_cache()

print("✅ GPU Memory Flushed. Ready to try again.")

In [ ]:
# Cell 3: FINAL STABLE TRAINING (Batch 4 Fix)
from ultralytics import YOLO
import torch

# Load Model
try:
    model = YOLO('yolo11m.pt')
except:
    model = YOLO('yolo11m.pt')

# Train
results = model.train(
    data='/kaggle/working/bigo_clean_config.yaml', 
    
    # --- MEMORY & STABILITY FIXES ---
    imgsz=1024,             # Keep High Res for Accuracy
    batch=4,                # <--- REDUCED to 4 to prevent OOM Crash
    amp=False,              # Keep False for OpenCV safety
    workers=0,              # Keep 0 for DataLoader safety
    # ------------------------------

    epochs=60,             
    patience=20,            
    
    optimizer='SGD',       
    lr0=0.01,
    momentum=0.937,
    
    augment=True,           
    name='BigO_Final_Batch4',
    project='/kaggle/working/runs/detect',
    val=True,
    save=True,
    exist_ok=True,
    device=0
)

In [ ]:
from ultralytics import YOLO

model = YOLO('/kaggle/input/bigoo/pytorch/default/1/last.pt') 

model.train(resume=True)

In [ ]:
from ultralytics import YOLO

model = YOLO('/kaggle/input/bigooo/pytorch/default/1/last.pt') 

model.train(resume=True)

In [ ]:
%pip install ultralytics seaborn matplotlib -q

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2
import os
import pandas as pd
import seaborn as sns

model_path = '/kaggle/working/runs/detect/BigO_Final_Batch4/weights/best.pt' 
model = YOLO(model_path)

config_path = '/kaggle/working/bigo_clean_config.yaml'

metrics = model.val(data=config_path, split='test', plots=True)

print(f"Mean Average Precision (mAP@50): {metrics.box.map50:.3f}")
print(f"Mean Average Precision (mAP@50-95): {metrics.box.map:.3f}")

In [ ]:
val_dir = metrics.save_dir
conf_matrix_path = os.path.join(val_dir, 'confusion_matrix_normalized.png')

plt.figure(figsize=(12, 10))
if os.path.exists(conf_matrix_path):
    img = cv2.imread(conf_matrix_path)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Confusion Matrix: Where the AI gets confused", fontsize=18, fontweight='bold')
    plt.axis('off')
    plt.show()
else:
    print("Confusion matrix file not found. Check your runs/detect/val folder.")

In [ ]:
import glob
import random
import os
import matplotlib.pyplot as plt
import cv2

test_images_path = '/kaggle/working/clean_dataset/test/images' 

test_files = []
for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
    test_files.extend(glob.glob(os.path.join(test_images_path, ext)))

print(f"Found {len(test_files)} test images.")

if len(test_files) == 0:
    print("Error: No images found! Check if the 'clean_dataset' folder exists.")
else:
    num_to_pick = min(len(test_files), 4)
    selected_files = random.sample(test_files, num_to_pick)

    plt.figure(figsize=(20, 10))

    for i, file in enumerate(selected_files):
        results = model.predict(file, conf=0.15, save=False)
        
        img_plot = results[0].plot() 
        
        plt.subplot(2, 2, i+1)
        plt.imshow(cv2.cvtColor(img_plot, cv2.COLOR_BGR2RGB))
        plt.title(f"Test Case #{i+1}\n({os.path.basename(file)})", fontsize=14)
        plt.axis('off')

    plt.suptitle(f"BigO Inference (Sampling {num_to_pick} Images)", fontsize=24)
    plt.tight_layout()
    plt.show()

In [ ]:
import glob
import random
import os
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. SETUP PATHS
# Ensure this points to your actual dataset inside clean_dataset
test_images_path = '/kaggle/working/clean_dataset/test/images'
best_weight_path = '/kaggle/working/runs/detect/BigO_Final_Batch4/weights/best.pt'

# 2. LOAD MODEL
# We load the best model you just trained
if not os.path.exists(best_weight_path):
    print("⚠️ Warning: best.pt not found. Using last.pt or checking paths...")
    best_weight_path = '/kaggle/working/runs/detect/BigO_Final_Batch4/weights/last.pt'

model = YOLO(best_weight_path)

# 3. GET IMAGES
# We grab all images to ensure we find some
test_files = []
for ext in ['*.jpg', '*.jpeg', '*.png']:
    test_files.extend(glob.glob(os.path.join(test_images_path, ext)))

# 4. SELECT THE BEST SAMPLES (Random but robust)
# If you have few images, take them all. If many, take 4.
num_samples = min(len(test_files), 4)
selected_files = random.sample(test_files, num_samples)

# 5. GENERATE THE "MONEY SHOT" PLOT
plt.figure(figsize=(24, 12))

for i, file in enumerate(selected_files):
    # --- THE TRICK FOR GREAT PLOTS ---
    # conf=0.35 : Hides weak guesses (removes clutter)
    # iou=0.5   : Cleans up overlapping boxes
    # line_width=2 : Makes boxes look sharp, not thick and messy
    results = model.predict(
        file, 
        conf=0.35,      # <--- TUNED FOR VISUALS
        iou=0.5, 
        save=False,
        verbose=False
    )
    
    # Plotting
    # We use built-in plot() with specific arguments for beauty
    res_plot = results[0].plot(line_width=3, font_size=1.2)
    
    plt.subplot(2, 2, i+1)
    plt.imshow(cv2.cvtColor(res_plot, cv2.COLOR_BGR2RGB))
    
    # Clean Title
    filename = os.path.basename(file)
    plt.title(f"Detection Result: {filename}", fontsize=18, fontweight='bold', color='#333333')
    plt.axis('off') # Remove ugly axis numbers

plt.suptitle(f"BigO AI Model Performance (Confidence > 35%)", fontsize=28, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig('/kaggle/working/BigO_Showcase_Result.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved High-Res Plot to: /kaggle/working/BigO_Showcase_Result.png")

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os
import glob
import random
import pandas as pd
import seaborn as sns
import numpy as np
from ultralytics import YOLO

# --- CONFIGURATION ---
# Path to your best trained weights
best_weight_path = '/kaggle/working/runs/detect/BigO_Final_Batch4/weights/best.pt'

# Path to your test images
test_images_path = '/kaggle/working/clean_dataset/test/images'

# Path to your data config used for validation
data_config_path = '/kaggle/working/bigo_clean_config.yaml'
# ---------------------

# 1. Load the trained model
if not os.path.exists(best_weight_path):
    print(f"⚠️ Warning: {best_weight_path} not found. Trying last.pt...")
    best_weight_path = best_weight_path.replace('best.pt', 'last.pt')

print(f"✅ Loading model from: {best_weight_path}")
model = YOLO(best_weight_path)

# 2. Get list of all test images
test_files = []
for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp']:
    test_files.extend(glob.glob(os.path.join(test_images_path, ext)))
print(f"✅ Found {len(test_files)} test images for evaluation.")

In [ ]:
# Run Validation on the Test Set
print("📊 Running Official Model Validation...")
# We set plots=True to automatically generate the confusion matrix and other standard plots
metrics = model.val(data=data_config_path, split='test', plots=True, verbose=False)

# --- DISPLAY KEY METRICS ---
print("\n" + "="*40)
print("   🏆 FINAL EVALUATION RESULTS   🏆")
print("="*40)
print(f"  mAP@50 (Accuracy):      {metrics.box.map50:.3f}  (Target: >0.90)")
print(f"  mAP@50-95 (Precision):  {metrics.box.map:.3f}   (Target: >0.60)")
print(f"  Precision (Few False+): {metrics.box.p.mean():.3f}")
print(f"  Recall (Few Misses):    {metrics.box.r.mean():.3f}")
print("="*40 + "\n")

# --- DISPLAY CONFUSION MATRIX ---
# YOLO saves plots in runs/detect/val/ (or similar)
val_dir = metrics.save_dir
conf_matrix_path = os.path.join(val_dir, 'confusion_matrix_normalized.png')

if os.path.exists(conf_matrix_path):
    plt.figure(figsize=(10, 8))
    img = cv2.imread(conf_matrix_path)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Normalized Confusion Matrix", fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("❌ Confusion matrix plot not found.")

In [ ]:
def plot_showcase_grid(model, image_files, title, conf=0.35, iou=0.5, rows=2, cols=2):
    """
    Generates a high-quality grid of model predictions.
    Conf=0.35 is the "Goldilocks" zone for looking good in presentations.
    """
    num_images = len(image_files)
    samples_to_plot = min(num_images, rows * cols)
    selected_files = random.sample(image_files, samples_to_plot)

    plt.figure(figsize=(cols * 6, rows * 5)) # Scale figure size with grid

    for i, file in enumerate(selected_files):
        # --- THE "SHOWCASE" PREDICTION ---
        # conf=0.35: Hides weak detections so plots look clean.
        # iou=0.5: Removes overlapping boxes for the same object.
        # line_width=3: Makes bounding boxes pop.
        results = model.predict(
            file, 
            conf=conf, 
            iou=iou, 
            save=False, 
            verbose=False
        )
        
        # Generate the plot with thick, clear lines
        res_plot = results[0].plot(line_width=3, font_size=1.1)
        
        plt.subplot(rows, cols, i+1)
        plt.imshow(cv2.cvtColor(res_plot, cv2.COLOR_BGR2RGB))
        
        filename = os.path.basename(file)
        plt.title(f"{filename}", fontsize=12)
        plt.axis('off')

    plt.suptitle(title, fontsize=20, fontweight='bold', y=0.99)
    plt.tight_layout()
    plt.show()

print("✅ Visualization function ready.")

In [ ]:
# --- GENERATE PLOT 1: General Performance ---
# This is your main "Look how good it is" slide.
plot_showcase_grid(
    model, 
    test_files, 
    title="BigO AI Model: Detection Performance on Unseen Data",
    conf=0.35, # Tuned for beauty
    rows=2, cols=2
)

# --- GENERATE PLOT 2: High Confidence Examples (Optional) ---
# Use this to show only the things the model is SUPER sure about.
# plot_showcase_grid(
#     model, 
#     test_files, 
#     title="High-Confidence Detections (Conf > 60%)",
#     conf=0.60, # Very strict threshold
#     rows=1, cols=3
# )

In [ ]:
# Cell 1: Install & Load SAHI
%pip install sahi ultralytics -q

from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import visualize_object_predictions
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2
import numpy as np
import os
import glob
import random

# 1. Define Path to your Best Weights
# Update this if your path is different
model_path = '/kaggle/input/m/aryankaushik22/bigoooo/pytorch/default/1/best.pt'

if not os.path.exists(model_path):
    print(f"⚠️ best.pt not found at {model_path}. Checking fallback...")
    # Fallback to last.pt if best isn't ready
    model_path = model_path.replace('best.pt', 'last.pt')

print(f"🚀 Loading Model into SAHI: {model_path}")

# 2. Initialize SAHI Model
# We use 'yolov8' model_type because YOLO11 is compatible with the same wrapper
detection_model = AutoDetectionModel.from_pretrained(
    model_type='yolov8',
    model_path=model_path,
    confidence_threshold=0.35, # Only accept confident detections
    device="cuda:0" # Use GPU
)

print("✅ SAHI Engine Ready.")

In [ ]:
# Cell 2: Compare Standard vs SAHI
# 1. Get Test Images
test_images_path = '/kaggle/working/clean_dataset/test/images'
test_files = glob.glob(os.path.join(test_images_path, '*.*'))

# 2. Pick a Random Image
image_path = random.choice(test_files)
image_numpy = cv2.imread(image_path)
image_numpy = cv2.cvtColor(image_numpy, cv2.COLOR_BGR2RGB) # Convert to RGB

print(f"Testing on: {os.path.basename(image_path)}")

# --- A. RUN STANDARD INFERENCE (Normal YOLO) ---
print("Running Standard Inference...")
base_model = YOLO(model_path)
standard_results = base_model.predict(image_path, conf=0.35, imgsz=1024, verbose=False)
standard_plot = standard_results[0].plot(line_width=2)

# --- B. RUN SAHI INFERENCE (Sliced) ---
print("Running SAHI Sliced Inference...")
sliced_result = get_sliced_prediction(
    image_path,
    detection_model,
    slice_height=512,    # Cut image into 512px chunks
    slice_width=512,
    overlap_height_ratio=0.2, # Overlap chunks to catch onions on the edge
    overlap_width_ratio=0.2
)

# Render SAHI Result
sahi_viz = visualize_object_predictions(
    image=np.array(image_numpy),
    object_prediction_list=sliced_result.object_prediction_list,
)

# --- C. VISUALIZE COMPARISON ---
plt.figure(figsize=(24, 12))

# Plot Standard
plt.subplot(1, 2, 1)
plt.imshow(standard_plot)
plt.title(f"Standard YOLO11\nDetections: {len(standard_results[0].boxes)}", fontsize=22, color='red', fontweight='bold')
plt.axis('off')

# Plot SAHI
plt.subplot(1, 2, 2)
plt.imshow(sahi_viz['image'])
# SAHI might find smaller objects
plt.title(f"BigO + SAHI (Sliced)\nDetections: {len(sliced_result.object_prediction_list)}", fontsize=22, color='green', fontweight='bold')
plt.axis('off')

plt.suptitle("Impact of SAHI Slicing on Detection", fontsize=28)
plt.tight_layout()
plt.show()

In [ ]:
# Cell: FAST FINE-TUNING (30 Epochs + Freeze)
from ultralytics import YOLO
import torch

# 1. Clear GPU Cache
torch.cuda.empty_cache()

# 2. Load your current BEST model
model_path = '/kaggle/input/m/aryankaushik22/bigoooo/pytorch/default/1/best.pt'
model = YOLO(model_path)

# 3. Train (Fast Mode)
results = model.train(
    data='/kaggle/working/bigo_clean_config.yaml', 
    
    # --- SPEED & TIME CONFIG ---
    epochs=30,              # REDUCED to 30 as requested
    freeze=10,              # CRITICAL: Freezes first 10 layers. Speeds up training by 40%.
    patience=10,            # Stop early if no improvement
    
    # --- ACCURACY SETTINGS ---
    imgsz=1024,             # Keep High Res (Required for Cuts)
    batch=4,                
    
    # --- THE "UNDERFITTING" CURE ---
    lr0=0.005,              
    lrf=0.01,               
    optimizer='SGD',        
    
    # --- AUGMENTATION BOOSTS ---
    hsv_h=0.015,            
    hsv_s=0.7,              # High Saturation for Sprouts
    hsv_v=0.4,              
    scale=0.5,              # Zoom for Cuts
    mosaic=1.0,
    
    # --- EXTRAS ---
    rect=True,              # Speed boost
    amp=False,              # Safety
    workers=0,              # Safety
    
    name='BigO_FineTune_Fast',
    project='/kaggle/working/runs/detect',
    val=True,
    save=True,
    exist_ok=True
)

In [2]:
%pip install "numpy<2.0" --force-reinstall

  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
datasets 4.4.1 requires pyarrow>=21.0.0, but you have pyarrow 19.0.1 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires notebook==6.5.7, but you have notebook 6.5.4 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.
google-colab 

In [1]:
%pip install ultralytics -q

import ipywidgets as widgets
from IPython.display import display
from ultralytics import YOLO
from PIL import Image, ImageDraw, ImageFont
import io
import numpy as np
import torch
import matplotlib.pyplot as plt

model_path = '/kaggle/input/sos/pytorch/default/1/best.pt' 

try:
    print(f"Loading model from: {model_path}...")
    model = YOLO(model_path)
    
    for class_id, class_name in list(model.names.items()):
        if class_name.lower() == 'fresh':
            model.names[class_id] = 'healthy'
    print(f"Classes: {model.names}")

except Exception as e:
    print(f"Error loading model: {e}")

uploader = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Upload Image'
)

output = widgets.Output()

def on_upload_change(change):
    with output:
        print("\n--- Processing Upload ---")
        if not uploader.value: return

        try:
            vals = uploader.value
            content = vals[0]['content'] if isinstance(vals, tuple) else list(vals.values())[0]['content']
            if hasattr(content, 'tobytes'): content = content.tobytes()

            pil_original = Image.open(io.BytesIO(content)).convert("RGB")
            
            pil_resized = pil_original.resize((640, 640), Image.Resampling.LANCZOS)
            
            img_np = np.array(pil_resized)
            img_tensor = torch.tensor(img_np)
            img_tensor = img_tensor.permute(2, 0, 1).unsqueeze(0).float() / 255.0

            results = model(img_tensor, verbose=False)
            det_count = len(results[0].boxes)
            print(f"Success! Found {det_count} objects.")

            draw = ImageDraw.Draw(pil_resized)
            
            try:
                font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 14)
            except:
                font = ImageFont.load_default()

            for result in results:
                for box in result.boxes:
                    cls_id = int(box.cls[0])
                    conf = float(box.conf[0])
                    label_name = model.names[cls_id]

                    color_rgb = (0, 0, 255) 
                    
                    if label_name == 'healthy':
                        color_rgb = (0, 255, 0)
                    elif 'sprout' in label_name.lower():
                        color_rgb = (255, 165, 0)
                    elif 'rot' in label_name.lower():
                        if conf > 0.60: 
                            color_rgb = (255, 0, 0)
                        else: 
                            color_rgb = (255, 165, 0)

                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    
                    draw.rectangle([x1, y1, x2, y2], outline=color_rgb, width=3)
                    
                    label_text = f" {label_name} {conf:.2f} "
                    
                    text_bbox = draw.textbbox((x1, y1), label_text, font=font)
                    text_width = text_bbox[2] - text_bbox[0]
                    text_height = text_bbox[3] - text_bbox[1]
                    
                    draw.rectangle(
                        [x1, y1 - text_height - 4, x1 + text_width, y1], 
                        fill=color_rgb
                    )
                    
                    draw.text((x1, y1 - text_height - 4), label_text, fill=(255, 255, 255), font=font)

            plt.figure(figsize=(10, 10))
            plt.imshow(pil_resized)
            plt.axis('off')
            plt.title(f"Result: {det_count} Objects Detected")
            plt.show()

        except Exception as e:
            print("ERROR:")
            print(e)
            import traceback
            traceback.print_exc()

        uploader.value = ()

uploader.observe(on_upload_change, names='value')
print("\nUploader Ready. Select an image:")
display(uploader, output)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 93.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 102.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 3.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 4.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

FileUpload(value=(), accept='image/*', description='Upload Image')

Output()